In [ ]:
!pip install streamlit pyngrok

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile banderas.py

import base64
import glob
import os
import unicodedata


def normalizar_nombre(nombre):

    nombre = unicodedata.normalize("NFD", nombre)

    nombre = "".join(
        caracter
        for caracter in nombre
        if unicodedata.category(caracter) != "Mn"
    )

    return nombre.lower().strip()


def cargar_banderas():

    banderas_encontradas = {}

    patrones = [
        "/content/drive/MyDrive/**/países/**/*.*",
        "/content/drive/MyDrive/**/paises/**/*.*"
    ]

    for patron in patrones:

        archivos = glob.glob(
            patron,
            recursive=True
        )

        for ruta in archivos:

            if not os.path.isfile(ruta):
                continue

            extension = ruta.lower().split(".")[-1]

            if extension == "jpg":
                extension = "jpeg"

            if extension not in ["png", "jpeg", "webp"]:
                continue

            nombre_archivo = os.path.splitext(
                os.path.basename(ruta)
            )[0]

            with open(ruta, "rb") as archivo:

                imagen_base64 = base64.b64encode(
                    archivo.read()
                ).decode()

            clave = normalizar_nombre(nombre_archivo)

            banderas_encontradas[clave] = (
                f"data:image/{extension};base64,{imagen_base64}"
            )

    return banderas_encontradas


def obtener_bandera_html(
    nombre_pais,
    banderas_img,
    css_class="bandera-partido"
):

    clave = normalizar_nombre(nombre_pais)
    imagen = banderas_img.get(clave, "")

    if not imagen:
        return ""

    return (
        f'<img '
        f'src="{imagen}" '
        f'class="{css_class}" '
        f'alt="Bandera de {nombre_pais}">'
    )

Overwriting banderas.py


In [ ]:
%%writefile data_loader.py

import json
import os
from functools import lru_cache


DEFAULT_DATA_DIR = (
    "/content/drive/MyDrive/"
    "WorldCupStoryteller/data"
)


class MatchDataError(RuntimeError):
    """Error controlado al cargar los datos de los partidos."""


def get_data_dir():

    return os.environ.get(
        "WORLDCUP_DATA_DIR",
        DEFAULT_DATA_DIR
    )


def read_json(file_path):

    if not os.path.exists(file_path):
        raise MatchDataError(
            f"No se encontró el archivo: {file_path}"
        )

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as file:

            return json.load(file)

    except json.JSONDecodeError as error:

        raise MatchDataError(
            f"JSON inválido en {file_path}: {error}"
        ) from error


@lru_cache(maxsize=1)
def load_match_index():

    index_path = os.path.join(
        get_data_dir(),
        "index.json"
    )

    index_data = read_json(
        index_path
    )

    matches = index_data.get(
        "matches",
        []
    )

    if len(matches) != 16:
        raise MatchDataError(
            "index.json debe contener exactamente "
            f"16 partidos y contiene {len(matches)}"
        )

    return index_data


@lru_cache(maxsize=32)
def load_match_package(slug):

    index_data = load_match_index()

    match_record = next(
        (
            match
            for match in index_data["matches"]
            if match["slug"] == slug
        ),
        None
    )

    if match_record is None:
        raise MatchDataError(
            f"No existe un partido con slug: {slug}"
        )

    relative_path = match_record[
        "file"
    ]

    package_path = os.path.join(
        get_data_dir(),
        relative_path
    )

    package = read_json(
        package_path
    )

    if package.get("slug") != slug:
        raise MatchDataError(
            f"El slug interno no coincide en {package_path}"
        )

    return package


def get_matches_by_stage(stage_key):

    index_data = load_match_index()

    return [
        match
        for match in index_data["matches"]
        if match["stage"]["key"] == stage_key
    ]


def clear_data_cache():

    load_match_index.cache_clear()
    load_match_package.cache_clear()

Overwriting data_loader.py


In [ ]:
# =========================================================
# PRUEBA DEL LECTOR DE DATOS
# =========================================================

from data_loader import (
    load_match_index,
    load_match_package
)


match_index = load_match_index()

france_argentina_data = (
    load_match_package(
        "france-vs-argentina"
    )
)


print(
    "✅ Índice cargado:",
    match_index["match_count"],
    "partidos"
)

print(
    "✅ Partido cargado:",
    france_argentina_data["match"]
)

print(
    "Estado:",
    france_argentina_data["status"]
)

print(
    "Resultado:",
    france_argentina_data["hero"]["score"]
)

print(
    "Tweets destacados:",
    len(
        france_argentina_data[
            "featured_tweets"
        ]["tweets"]
    )
)

✅ Índice cargado: 16 partidos
✅ Partido cargado: France vs Argentina
Estado: ready
Resultado: 4 - 3
Tweets destacados: 3


In [ ]:
%%writefile app_config.py

# =========================================================
# NOMBRES VISIBLES DE LOS EQUIPOS
# =========================================================

TEAM_NAMES_ES = {
    "France": "Francia",
    "Argentina": "Argentina",
    "Uruguay": "Uruguay",
    "Portugal": "Portugal",
    "Spain": "España",
    "Russia": "Rusia",
    "Croatia": "Croacia",
    "Denmark": "Dinamarca",
    "Brazil": "Brasil",
    "Mexico": "México",
    "Belgium": "Bélgica",
    "Japan": "Japón",
    "Sweden": "Suecia",
    "Switzerland": "Suiza",
    "Colombia": "Colombia",
    "England": "Inglaterra"
}


# =========================================================
# ORDEN Y PRESENTACIÓN DE LAS FASES
# =========================================================

STAGE_CONFIG = {
    "octavos": {
        "stage_key": "round_of_16",
        "nombre": "Octavos de final",
        "cantidad": "8 partidos"
    },
    "cuartos": {
        "stage_key": "quarterfinals",
        "nombre": "Cuartos de final",
        "cantidad": "4 partidos"
    },
    "semifinales": {
        "stage_key": "semifinals",
        "nombre": "Semifinales",
        "cantidad": "2 partidos"
    },
    "tercer-puesto": {
        "stage_key": "third_place",
        "nombre": "Tercer puesto",
        "cantidad": "1 partido"
    },
    "final": {
        "stage_key": "final",
        "nombre": "Final",
        "cantidad": "1 partido"
    }
}


def get_team_name_es(team_name):

    return TEAM_NAMES_ES.get(
        team_name,
        team_name
    )

Overwriting app_config.py


In [ ]:
%%writefile chart_config.py

# =========================================================
# COLORES DE LAS SELECCIONES
# =========================================================

TEAM_COLORS = {
    "France": "#2457D6",
    "Argentina": "#75AADB",
    "Uruguay": "#55BDE6",
    "Portugal": "#C92535",
    "Brazil": "#F7D117",
    "Mexico": "#168B52",
    "Belgium": "#E12C3D",
    "Japan": "#1D4E9E",
    "Spain": "#AA151B",
    "Russia": "#F4F4F4",
    "Croatia": "#F4F4F4",
    "Denmark": "#C8102E",
    "Sweden": "#1769AA",
    "Switzerland": "#D52B1E",
    "Colombia": "#FCD116",
    "England": "#F4F4F4"
}


MATCH_COLOR_OVERRIDES = {
    "Russia vs Croatia": {
        "Russia": "#F4F4F4",
        "Croatia": "#171717"
    },
    "Croatia vs England": {
        "Croatia": "#171717",
        "England": "#F4F4F4"
    }
}


# =========================================================
# EQUIPO DE CADA EVENTO HISTÓRICO
# =========================================================

EVENT_TEAM_ORDER = {
    "France vs Argentina": [
        "France",
        "Argentina",
        "Argentina",
        "France",
        "France",
        "France",
        "Argentina"
    ],
    "Uruguay vs Portugal": [
        "Uruguay",
        "Portugal",
        "Uruguay"
    ],
    "Spain vs Russia": [
        "Spain",
        "Russia",
        "Russia"
    ],
    "Croatia vs Denmark": [
        "Denmark",
        "Croatia",
        "Denmark",
        "Croatia"
    ],
    "Brazil vs Mexico": [
        "Brazil",
        "Brazil"
    ],
    "Belgium vs Japan": [
        "Japan",
        "Japan",
        "Belgium",
        "Belgium",
        "Belgium"
    ],
    "Sweden vs Switzerland": [
        "Sweden"
    ],
    "Colombia vs England": [
        "England",
        "Colombia",
        "England"
    ],
    "Uruguay vs France": [
        "France",
        "France"
    ],
    "Brazil vs Belgium": [
        "Belgium",
        "Belgium",
        "Brazil"
    ],
    "Russia vs Croatia": [
        "Russia",
        "Croatia",
        "Croatia",
        "Russia",
        "Croatia"
    ],
    "Sweden vs England": [
        "England",
        "England"
    ],
    "France vs Belgium": [
        "France"
    ],
    "Croatia vs England": [
        "England",
        "Croatia",
        "Croatia"
    ],
    "Belgium vs England (3rd Place)": [
        "Belgium",
        "Belgium"
    ],
    "France vs Croatia": [
        "France",
        "Croatia",
        "France",
        "France",
        "France",
        "Croatia"
    ]
}


EVENT_TYPE_OVERRIDES = {
    ("Spain vs Russia", 120): "shootout",
    ("Croatia vs Denmark", 116): "save",
    ("Croatia vs Denmark", 120): "shootout",
    ("Colombia vs England", 120): "shootout",
    ("Russia vs Croatia", 120): "shootout"
}


def hex_to_rgba(
    hex_color,
    opacity
):

    color = hex_color.lstrip("#")

    red = int(color[0:2], 16)
    green = int(color[2:4], 16)
    blue = int(color[4:6], 16)

    return (
        f"rgba("
        f"{red}, {green}, {blue}, {opacity}"
        f")"
    )


def get_visible_line_color(color):

    if color.upper() == "#171717":
        return "#A3A3A3"

    if color.upper() == "#F4F4F4":
        return "#FFFFFF"

    return color


def get_match_colors(
    match_name,
    team1,
    team2
):

    colors = TEAM_COLORS.copy()

    colors.update(
        MATCH_COLOR_OVERRIDES.get(
            match_name,
            {}
        )
    )

    return (
        colors.get(team1, "#2457D6"),
        colors.get(team2, "#75AADB")
    )


def get_event_team(
    match_name,
    event_index,
    event
):

    if event.get("team"):
        return event["team"]

    teams = EVENT_TEAM_ORDER.get(
        match_name,
        []
    )

    if event_index < len(teams):
        return teams[event_index]

    return None


def get_event_type(
    match_name,
    event
):

    if event.get("type"):
        return event["type"]

    event_key = (
        match_name,
        int(event["minute"])
    )

    if event_key in EVENT_TYPE_OVERRIDES:
        return EVENT_TYPE_OVERRIDES[
            event_key
        ]

    event_text = (
        event.get("event", "")
        .lower()
    )

    if "ataja" in event_text:
        return "save"

    if (
        "penales" in event_text
        or "tanda" in event_text
        or "clasifica" in event_text
    ):
        return "shootout"

    return "goal"

Overwriting chart_config.py


In [ ]:
%%writefile match_charts.py

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

from app_config import get_team_name_es
from banderas import normalizar_nombre

from chart_config import (
    get_event_team,
    get_event_type,
    get_match_colors,
    get_visible_line_color,
    hex_to_rgba
)


def get_flag_uri(
    team_name,
    banderas_img
):

    visible_name = get_team_name_es(
        team_name
    )

    flag_key = normalizar_nombre(
        visible_name
    )

    return banderas_img.get(
        flag_key,
        ""
    )


def get_missing_ranges(
    momentum
):

    missing = momentum.loc[
        ~momentum["has_coverage"],
        [
            "display_start",
            "display_end"
        ]
    ]

    if missing.empty:
        return []

    ranges = []

    current_start = float(
        missing.iloc[0][
            "display_start"
        ]
    )

    current_end = float(
        missing.iloc[0][
            "display_end"
        ]
    )

    for _, row in missing.iloc[1:].iterrows():

        start = float(
            row["display_start"]
        )

        end = float(
            row["display_end"]
        )

        if np.isclose(
            start,
            current_end
        ):
            current_end = end

        else:
            ranges.append(
                (
                    current_start,
                    current_end
                )
            )

            current_start = start
            current_end = end

    ranges.append(
        (
            current_start,
            current_end
        )
    )

    return ranges


def football_minute_to_display(
    minute,
    periods
):

    minute = float(minute)

    play_periods = [
        period
        for period in periods
        if period["type"] == "play"
    ]

    for period in play_periods:

        football_start = float(
            period["football_start"]
        )

        football_end = float(
            period["football_end"]
        )

        if (
            football_start
            <= minute
            <= football_end
        ):

            football_duration = (
                football_end
                - football_start
            )

            if football_duration <= 0:
                return None

            progress = (
                minute
                - football_start
            ) / football_duration

            return (
                float(
                    period["display_start"]
                )
                + progress
                * float(
                    period["display_width"]
                )
            )

    if play_periods:

        last_period = play_periods[-1]

        if minute >= float(
            last_period["football_end"]
        ):
            return float(
                last_period["display_end"]
            )

    return None


def shootout_order_to_display(
    order,
    periods,
    shootout
):

    shootout_period = next(
        (
            period
            for period in periods
            if period["key"] == "shootout"
        ),
        None
    )

    if shootout_period is None:
        return None

    total_penalties = max(
        len(shootout),
        1
    )

    start_x = float(
        shootout_period["display_start"]
    )

    width = float(
        shootout_period["display_width"]
    )

    return (
        start_x
        + (
            float(order)
            / (total_penalties + 1)
        )
        * width
    )


def build_final_axis(
    metadata
):

    tick_values = []
    tick_text = []
    guide_values = []

    for period in metadata["periods"]:

        start_x = float(
            period["display_start"]
        )

        end_x = float(
            period["display_end"]
        )

        if period["type"] == "play":

            football_start = float(
                period["football_start"]
            )

            football_end = float(
                period["football_end"]
            )

            football_duration = (
                football_end
                - football_start
            )

            if football_duration == 45:

                football_ticks = list(
                    np.arange(
                        football_start,
                        football_end + 0.1,
                        15
                    )
                )

            else:

                football_ticks = [
                    football_start,
                    football_end
                ]

            for football_minute in (
                football_ticks
            ):

                progress = (
                    football_minute
                    - football_start
                ) / football_duration

                display_x = (
                    start_x
                    + progress
                    * float(
                        period["display_width"]
                    )
                )

                tick_values.append(
                    display_x
                )

                tick_text.append(
                    f"{int(football_minute)}′"
                )

                guide_values.append(
                    display_x
                )

        else:

            center_x = (
                start_x + end_x
            ) / 2

            if (
                period["key"]
                == "extra_time_transition"
            ):

                label = (
                    "<b>TIEMPO<br>"
                    "SUPLEMENTARIO</b>"
                )

            else:

                label = (
                    f"<b>{period['label']}</b>"
                )

            tick_values.append(
                center_x
            )

            tick_text.append(
                label
            )

    return (
        tick_values,
        tick_text,
        guide_values
    )


def render_momentum_unavailable(
    match_data
):

    quality = match_data[
        "data_quality"
    ]

    st.html(
        f"""
        <section style="
            padding: 95px 7%;
            background: #050505;
            color: white;
            text-align: center;
        ">
            <div style="
                max-width: 760px;
                margin: 0 auto;
            ">
                <div style="
                    color: #d4af37;
                    font-size: 13px;
                    font-weight: 900;
                    letter-spacing: 2px;
                ">
                    PULSO DEL PARTIDO EN TWITTER
                </div>

                <h2 style="
                    margin: 18px 0;
                    font-size: 38px;
                ">
                    Sin línea temporal disponible
                </h2>

                <p style="
                    color: rgba(255,255,255,0.68);
                    font-size: 17px;
                    line-height: 1.6;
                ">
                    {quality["message"]}
                </p>
            </div>
        </section>
        """
    )


def render_twitter_momentum(
    match_data,
    banderas_img
):

    quality = match_data[
        "data_quality"
    ]

    if not quality.get(
        "show_momentum",
        False
    ):

        render_momentum_unavailable(
            match_data
        )

        return

    momentum_data = match_data[
        "momentum"
    ]

    metadata = momentum_data[
        "metadata"
    ]

    momentum = pd.DataFrame(
        momentum_data["series"]
    )

    if momentum.empty:

        render_momentum_unavailable(
            match_data
        )

        return

    for column in [
        "minute",
        "display_start",
        "display_end",
        "team1_mentions",
        "team2_mentions",
        "team2_plot"
    ]:

        momentum[column] = pd.to_numeric(
            momentum[column],
            errors="coerce"
        )

    momentum["has_coverage"] = (
        momentum["has_coverage"]
        .fillna(False)
        .astype(bool)
    )

    match_name = match_data["match"]

    team1 = metadata["team1"]
    team2 = metadata["team2"]

    (
        team1_color,
        team2_color
    ) = get_match_colors(
        match_name,
        team1,
        team2
    )

    team1_line = get_visible_line_color(
        team1_color
    )

    team2_line = get_visible_line_color(
        team2_color
    )

    available_mentions = pd.concat([
        momentum["team1_mentions"],
        momentum["team2_mentions"]
    ]).dropna()

    if (
        available_mentions.empty
        or available_mentions.max() <= 0
    ):
        max_mentions = 1.0

    else:
        max_mentions = float(
            available_mentions.max()
        )

    vertical_limit = (
        max_mentions * 1.30
    )

    figure = go.Figure()

    figure.add_trace(
        go.Scatter(
            x=momentum["minute"],
            y=momentum[
                "team1_mentions"
            ],
            mode="lines",
            line={
                "color": team1_line,
                "width": 3,
                "shape": "spline",
                "smoothing": 1.10
            },
            fill="tozeroy",
            fillcolor=hex_to_rgba(
                team1_color,
                0.76
            ),
            connectgaps=False,
            name=team1,
            customdata=np.column_stack([
                momentum[
                    "period_label"
                ],
                momentum[
                    "team1_mentions"
                ]
            ]),
            hovertemplate=(
                f"<b>{team1}</b><br>"
                "%{customdata[0]}<br>"
                "%{customdata[1]:.0f} menciones"
                "<extra></extra>"
            )
        )
    )

    figure.add_trace(
        go.Scatter(
            x=momentum["minute"],
            y=momentum["team2_plot"],
            mode="lines",
            line={
                "color": team2_line,
                "width": 3,
                "shape": "spline",
                "smoothing": 1.10
            },
            fill="tozeroy",
            fillcolor=hex_to_rgba(
                team2_color,
                0.76
            ),
            connectgaps=False,
            name=team2,
            customdata=np.column_stack([
                momentum[
                    "period_label"
                ],
                momentum[
                    "team2_mentions"
                ]
            ]),
            hovertemplate=(
                f"<b>{team2}</b><br>"
                "%{customdata[0]}<br>"
                "%{customdata[1]:.0f} menciones"
                "<extra></extra>"
            )
        )
    )

    period_colors = {
        "halftime":
            "rgba(212,175,55,0.12)",

        "extra_time_transition":
            "rgba(29,78,158,0.17)",

        "extra_time_halftime":
            "rgba(212,175,55,0.12)",

        "shootout":
            "rgba(213,43,30,0.13)"
    }

    for period in metadata["periods"]:

        if period["type"] == "play":
            continue

        figure.add_vrect(
            x0=period["display_start"],
            x1=period["display_end"],
            fillcolor=period_colors.get(
                period["key"],
                "rgba(255,255,255,0.08)"
            ),
            line_width=0,
            layer="below"
        )

    for start, end in (
        get_missing_ranges(
            momentum
        )
    ):

        figure.add_vrect(
            x0=start,
            x1=end,
            fillcolor=(
                "rgba(0,0,0,0.17)"
            ),
            line_width=0,
            layer="below"
        )

        if end - start >= 20:

            figure.add_annotation(
                x=(start + end) / 2,
                y=0,
                text=(
                    "SIN COBERTURA "
                    "DEL DATASET"
                ),
                textangle=-90,
                showarrow=False,
                font={
                    "size": 10,
                    "color":
                        "rgba(255,255,255,0.30)"
                }
            )

    figure.add_hline(
        y=0,
        line_color="#F2F2F2",
        line_width=2
    )

    icons = {
        "goal": "⚽",
        "save": "🧤"
    }

    event_distance = (
        max_mentions * 0.18
    )

    for event_index, event in enumerate(
        metadata["events"]
    ):

        event_type = get_event_type(
            match_name,
            event
        )

        if (
            event_type == "shootout"
            and metadata["has_shootout"]
        ):
            continue

        event_x = football_minute_to_display(
            event["minute"],
            metadata["periods"]
        )

        if event_x is None:
            continue

        event_team = get_event_team(
            match_name,
            event_index,
            event
        )

        direction = (
            1
            if event_team == team1
            else -1
        )

        event_y = (
            direction
            * event_distance
            * (
                1
                + 0.16
                * (event_index % 2)
            )
        )

        figure.add_annotation(
            x=event_x,
            y=event_y,
            ax=event_x,
            ay=0,
            xref="x",
            yref="y",
            axref="x",
            ayref="y",
            text="",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor="#F2F2F2"
        )

        figure.add_trace(
            go.Scatter(
                x=[event_x],
                y=[event_y],
                mode="text",
                text=[
                    icons.get(
                        event_type,
                        "●"
                    )
                ],
                textfont={
                    "size": 17,
                    "color": "#FFFFFF"
                },
                showlegend=False,
                hovertemplate=(
                    f"<b>{int(event['minute'])}′</b><br>"
                    f"{event.get('event', '')}"
                    "<extra></extra>"
                )
            )
        )

    shootout = metadata.get(
        "shootout",
        []
    )

    penalty_distance = (
        max_mentions * 0.28
    )

    for penalty in shootout:

        penalty_x = (
            shootout_order_to_display(
                penalty["order"],
                metadata["periods"],
                shootout
            )
        )

        if penalty_x is None:
            continue

        penalty_y = (
            penalty_distance
            if penalty["team"] == team1
            else -penalty_distance
        )

        penalty_icon = (
            "⚽"
            if penalty["outcome"] == "scored"
            else "✕"
        )

        figure.add_trace(
            go.Scatter(
                x=[penalty_x],
                y=[penalty_y],
                mode="text",
                text=[penalty_icon],
                textfont={
                    "size": 19,
                    "color": "#FFFFFF"
                },
                showlegend=False,
                hovertemplate=(
                    f"<b>{penalty.get('player', '')}</b><br>"
                    f"{penalty.get('team', '')}"
                    "<extra></extra>"
                )
            )
        )

    (
        tick_values,
        tick_text,
        guide_values
    ) = build_final_axis(
        metadata
    )

    for guide_x in guide_values:

        figure.add_vline(
            x=guide_x,
            line_color=(
                "rgba(255,255,255,0.15)"
            ),
            line_width=1,
            line_dash="dot",
            layer="below"
        )

    figure.update_layout(
        title={
            "text": (
                "<b>"
                "PULSO DEL PARTIDO EN TWITTER"
                "</b>"
                "<br>"
                "<span style='font-size:15px;'>"
                "Menciones por equipo "
                "en intervalos de cinco minutos"
                "</span>"
            ),
            "x": 0.5,
            "xanchor": "center",
            "font": {
                "size": 29,
                "color": "#FFFFFF"
            }
        },
        height=650,
        margin={
            "l": 125,
            "r": 45,
            "t": 145,
            "b": 105
        },
        paper_bgcolor="#050505",
        plot_bgcolor="#3B3F41",
        font={
            "color": "#FFFFFF"
        },
        showlegend=False,
        hovermode="x",
        xaxis={
            "range": [
                0,
                metadata["display_end"]
            ],
            "tickmode": "array",
            "tickvals": tick_values,
            "ticktext": tick_text,
            "tickfont": {
                "size": (
                    10
                    if metadata[
                        "has_extra_time"
                    ]
                    else 12
                ),
                "color": "#FFFFFF"
            },
            "showgrid": False,
            "showline": True,
            "linecolor": "#FFFFFF",
            "zeroline": False,
            "automargin": True
        },
        yaxis={
            "range": [
                -vertical_limit,
                vertical_limit
            ],
            "showticklabels": False,
            "showgrid": False,
            "showline": True,
            "linecolor": "#FFFFFF",
            "zeroline": False
        }
    )

    team1_flag = get_flag_uri(
        team1,
        banderas_img
    )

    team2_flag = get_flag_uri(
        team2,
        banderas_img
    )

    if team1_flag:

        figure.add_layout_image({
            "source": team1_flag,
            "xref": "paper",
            "yref": "paper",
            "x": -0.085,
            "y": 0.76,
            "sizex": 0.075,
            "sizey": 0.075,
            "xanchor": "center",
            "yanchor": "middle",
            "sizing": "contain",
            "layer": "above"
        })

    if team2_flag:

        figure.add_layout_image({
            "source": team2_flag,
            "xref": "paper",
            "yref": "paper",
            "x": -0.085,
            "y": 0.24,
            "sizex": 0.075,
            "sizey": 0.075,
            "xanchor": "center",
            "yanchor": "middle",
            "sizing": "contain",
            "layer": "above"
        })

    frame_left = -0.17
    frame_right = 1.06
    frame_bottom = -0.27
    frame_top = 1.38

    figure.add_shape(
        type="rect",
        x0=frame_left,
        x1=frame_right,
        y0=frame_bottom,
        y1=frame_top,
        line={
            "color": "#D4AF37",
            "width": 6
        },
        fillcolor="rgba(0,0,0,0)",
        xref="paper",
        yref="paper",
        layer="above"
    )

    st.plotly_chart(
        figure,
        use_container_width=True,
        config={
            "displayModeBar": False,
            "responsive": True
        }
    )

Overwriting match_charts.py


In [ ]:
%%writefile match_components.py

import streamlit as st
from html import escape
from textwrap import dedent

from app_config import get_team_name_es
from banderas import obtener_bandera_html


def format_number_es(value):

    return (
        f"{int(value):,}"
        .replace(",", ".")
    )


def render_match_hero(
    match_data,
    banderas_img,
    fase_slug
):

    hero = match_data["hero"]
    quality = match_data["data_quality"]

    team1 = hero["team1"]
    team2 = hero["team2"]

    team1_name = get_team_name_es(
        team1["name"]
    )

    team2_name = get_team_name_es(
        team2["name"]
    )

    winner_name = get_team_name_es(
        hero["winner"]
    )

    team1_flag = obtener_bandera_html(
        team1_name,
        banderas_img,
        css_class="hero-flag"
    )

    team2_flag = obtener_bandera_html(
        team2_name,
        banderas_img,
        css_class="hero-flag"
    )

    tweet_count = format_number_es(
        hero["tweet_count"]
    )

    stage_label = hero["stage"]["label"]
    date_display = hero["date"]["display"]

    team1_score = team1.get(
        "score",
        "-"
    )

    team2_score = team2.get(
        "score",
        "-"
    )

    st.markdown(
        dedent("""
        <style>

        .match-hero {
            position: relative;
            min-height: 92vh;
            box-sizing: border-box;
            overflow: hidden;

            display: flex;
            flex-direction: column;
            justify-content: center;

            padding: 48px 7% 72px;

            color: #ffffff;

            background:
                radial-gradient(
                    circle at 50% 20%,
                    rgba(38, 91, 126, 0.72) 0%,
                    rgba(8, 27, 41, 0.96) 48%,
                    #020609 100%
                );
        }

        .match-hero::before {
            content: "";
            position: absolute;
            inset: 0;

            background:
                linear-gradient(
                    115deg,
                    rgba(213, 43, 30, 0.10),
                    transparent 34%
                ),
                linear-gradient(
                    245deg,
                    rgba(29, 78, 158, 0.12),
                    transparent 34%
                );

            pointer-events: none;
        }

        .hero-worldcup-line {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            height: 9px;

            background:
                linear-gradient(
                    90deg,
                    #d52b1e 0%,
                    #d52b1e 34%,
                    #1d4e9e 34%,
                    #1d4e9e 68%,
                    #d4af37 68%,
                    #d4af37 100%
                );
        }

        .hero-content {
            position: relative;
            z-index: 2;

            width: 100%;
            max-width: 1180px;
            margin: 0 auto;
        }

        .hero-back {
            display: inline-block;
            margin-bottom: 55px;

            color: #8ed8f8 !important;
            text-decoration: none !important;

            font-size: 15px;
            font-weight: 750;
        }

        .hero-back:hover {
            color: #ffffff !important;
        }

        .hero-kicker {
            text-align: center;
            margin-bottom: 36px;

            color: #d4af37;

            font-size: 13px;
            font-weight: 850;
            letter-spacing: 2.2px;
            text-transform: uppercase;
        }

        .hero-scoreboard {
            display: grid;
            grid-template-columns:
                minmax(220px, 1fr)
                minmax(210px, 0.65fr)
                minmax(220px, 1fr);

            align-items: center;
            gap: 34px;
        }

        .hero-team {
            display: flex;
            flex-direction: column;
            align-items: center;

            text-align: center;
        }

        .hero-flag {
            width: 112px;
            height: 74px;

            object-fit: cover;

            border-radius: 7px;
            border: 1px solid rgba(255,255,255,0.28);

            box-shadow:
                0 12px 30px rgba(0,0,0,0.34);
        }

        .hero-team-name {
            margin-top: 20px;

            color: #ffffff;

            font-size: clamp(27px, 3vw, 43px);
            font-weight: 850;
            line-height: 1.05;
        }

        .hero-result {
            text-align: center;
        }

        .hero-score {
            color: #ffffff;

            font-size: clamp(65px, 9vw, 112px);
            font-weight: 900;
            letter-spacing: -6px;
            line-height: 0.95;

            text-shadow:
                0 8px 30px rgba(0,0,0,0.32);
        }

        .hero-score-separator {
            color: rgba(255,255,255,0.45);
            margin: 0 14px;
        }

        .hero-winner {
            margin-top: 22px;

            color: rgba(255,255,255,0.74);

            font-size: 13px;
            font-weight: 750;
            letter-spacing: 1.1px;
            text-transform: uppercase;
        }

        .hero-metrics {
            display: flex;
            justify-content: center;
            flex-wrap: wrap;
            gap: 12px;

            margin-top: 58px;
        }

        .hero-pill {
            padding: 11px 17px;

            border-radius: 999px;
            border: 1px solid rgba(255,255,255,0.16);

            background: rgba(255,255,255,0.075);

            color: rgba(255,255,255,0.86);

            font-size: 12px;
            font-weight: 800;
            letter-spacing: 0.8px;
            text-transform: uppercase;
        }

        @media (max-width: 760px) {

            .match-hero {
                min-height: auto;
                padding:
                    34px 22px
                    60px;
            }

            .hero-back {
                margin-bottom: 38px;
            }

            .hero-scoreboard {
                grid-template-columns:
                    1fr 0.8fr 1fr;

                gap: 10px;
            }

            .hero-flag {
                width: 72px;
                height: 48px;
            }

            .hero-team-name {
                font-size: 20px;
            }

            .hero-score {
                font-size: 52px;
                letter-spacing: -4px;
            }

            .hero-score-separator {
                margin: 0 5px;
            }

            .hero-winner {
                font-size: 10px;
            }

            .hero-metrics {
                margin-top: 42px;
            }
        }

        </style>
        """),
        unsafe_allow_html=True
    )

    hero_html = f"""
    <section class="match-hero">

        <div class="hero-worldcup-line"></div>

        <div class="hero-content">

            <a
                href="?p=fases&amp;fase={escape(fase_slug)}"
                target="_self"
                class="hero-back"
            >
                ← Volver a la fase
            </a>

            <div class="hero-kicker">
                {escape(stage_label)}
                &nbsp;•&nbsp;
                {escape(date_display)}
            </div>

            <div class="hero-scoreboard">

                <div class="hero-team">
                    {team1_flag}

                    <div class="hero-team-name">
                        {escape(team1_name)}
                    </div>
                </div>

                <div class="hero-result">

                    <div class="hero-score">
                        {team1_score}
                        <span class="hero-score-separator">-</span>
                        {team2_score}
                    </div>

                    <div class="hero-winner">
                        Ganador: {escape(winner_name)}
                    </div>
                </div>

                <div class="hero-team">
                    {team2_flag}

                    <div class="hero-team-name">
                        {escape(team2_name)}
                    </div>
                </div>

            </div>

            <div class="hero-metrics">

                <div class="hero-pill">
                    {tweet_count} tuits analizados
                </div>

                <div class="hero-pill">
                    {escape(quality["label"])}
                </div>

            </div>

        </div>

    </section>
    """

    st.html(
    dedent(
        hero_html
    ).strip()
)
def render_match_story(
    match_data
):

    story = match_data["story"]

    story_blocks = [
        story["preview"],
        story["social_climate"],
        story["turning_point"],
        story["outcome"]
    ]

    story_rows = ""

    for position, block in enumerate(
        story_blocks,
        start=1
    ):

        story_rows += f"""
        <div class="story-row">

            <div class="story-chapter">

                <span class="story-number">
                    {position:02d}
                </span>

                <h3>
                    {escape(block["title"])}
                </h3>

            </div>

            <div class="story-text">
                {escape(block["text"])}
            </div>

        </div>
        """

    coverage_note = story.get(
        "coverage_note",
        ""
    )

    story_html = f"""
    <section class="match-story">

        <div class="story-container">

            <div class="story-heading">

                <div class="story-eyebrow">
                    LA HISTORIA DEL PARTIDO
                </div>

                <h2>
                    Así lo vivió Twitter
                </h2>

                <p>
                    Una reconstrucción narrativa a partir
                    de la conversación social.
                </p>

            </div>

            <div class="story-article">
                {story_rows}
            </div>

            <div class="story-note">
                {escape(coverage_note)}
            </div>

        </div>

    </section>
    """

    st.html(
        """
        <style>

        .match-story {
            box-sizing: border-box;

            padding: 105px 7% 115px;

            background:
                linear-gradient(
                    180deg,
                    #f5f1e9 0%,
                    #fffdf9 100%
                );

            color: #1d252b;
        }

        .story-container {
            max-width: 1040px;
            margin: 0 auto;
        }

        .story-heading {
            max-width: 760px;
            margin-bottom: 72px;
        }

        .story-eyebrow {
            margin-bottom: 17px;

            color: #9e3545;

            font-size: 13px;
            font-weight: 900;
            letter-spacing: 2.2px;
            text-transform: uppercase;
        }

        .story-heading h2 {
            margin: 0 0 18px;

            color: #20272d;

            font-size: clamp(38px, 5vw, 63px);
            font-weight: 900;
            letter-spacing: -2px;
            line-height: 1.02;
        }

        .story-heading p {
            margin: 0;

            color: #706d68;

            font-size: 19px;
            line-height: 1.6;
        }

        .story-article {
            border-top:
                1px solid rgba(32,39,45,0.17);
        }

        .story-row {
            display: grid;

            grid-template-columns:
                minmax(210px, 0.68fr)
                minmax(0, 1.7fr);

            gap: 55px;

            padding: 38px 0 42px;

            border-bottom:
                1px solid rgba(32,39,45,0.17);
        }

        .story-chapter {
            display: grid;

            grid-template-columns:
                42px 1fr;

            gap: 16px;
            align-items: start;
        }

        .story-number {
            color: #c0a052;

            font-size: 12px;
            font-weight: 900;
            letter-spacing: 1px;
        }

        .story-chapter h3 {
            margin: -4px 0 0;

            color: #792d3b;

            font-size: 21px;
            font-weight: 850;
            line-height: 1.18;
        }

        .story-text {
            color: #343a3e;

            font-family:
                Georgia,
                "Times New Roman",
                serif;

            font-size: 20px;
            line-height: 1.78;
        }

        .story-note {
            margin-top: 28px;

            color: #85817b;

            font-size: 12px;
            line-height: 1.5;
            font-style: italic;
        }

        @media (max-width: 760px) {

            .match-story {
                padding:
                    75px 24px
                    85px;
            }

            .story-heading {
                margin-bottom: 50px;
            }

            .story-row {
                grid-template-columns: 1fr;
                gap: 22px;

                padding: 30px 0 34px;
            }

            .story-text {
                font-size: 18px;
                line-height: 1.68;
            }
        }

        </style>
        """
    )

    st.html(
        dedent(
            story_html
        ).strip()
    )

Overwriting match_components.py


In [ ]:
%%writefile -a match_components.py

# =========================================================
# QUIÉN DOMINÓ LA CONVERSACIÓN
# =========================================================

from banderas import obtener_bandera_html
from app_config import get_team_name_es
from chart_config import get_match_colors


def render_share_of_voice(
    match_data,
    banderas_img
):
    """
    Muestra el Share of Voice como una barra horizontal
    dividida entre ambos equipos.
    """

    share = match_data.get(
        "share_of_voice",
        {}
    )

    quality = match_data.get(
        "data_quality",
        {}
    )

    team1_data = share.get(
        "team1",
        {}
    )

    team2_data = share.get(
        "team2",
        {}
    )

    team1 = team1_data.get(
        "name",
        ""
    )

    team2 = team2_data.get(
        "name",
        ""
    )

    team1_es = get_team_name_es(
        team1
    )

    team2_es = get_team_name_es(
        team2
    )

    # -----------------------------------------------------
    # SECCIÓN SIN INFORMACIÓN SOCIAL
    # -----------------------------------------------------

    if (
    not quality.get(
        "show_social_analysis",
        False
    )
    or not share.get(
        "available",
        False
    )
):

      render_unavailable_section(
        kicker="LA CONVERSACIÓN",
        title="Quién dominó la conversación",
        message=(
            "No hay suficientes menciones para comparar "
            "la conversación entre ambos equipos."
        ),
        theme="light"
    )

      return

    # -----------------------------------------------------
    # DATOS
    # -----------------------------------------------------

    percentage1 = float(
        team1_data.get(
            "percentage",
            0
        )
    )

    percentage2 = float(
        team2_data.get(
            "percentage",
            0
        )
    )

    mentions1 = int(
        team1_data.get(
            "mentions",
            0
        )
    )

    mentions2 = int(
        team2_data.get(
            "mentions",
            0
        )
    )

    color1, color2 = get_match_colors(
        match_data.get(
            "match",
            ""
        ),
        team1,
        team2
    )

    flag1 = obtener_bandera_html(
        team1_es,
        banderas_img,
        css_class="sov-flag"
    )

    flag2 = obtener_bandera_html(
        team2_es,
        banderas_img,
        css_class="sov-flag"
    )

    interpretation = share.get(
        "interpretation",
        ""
    )

    # -----------------------------------------------------
    # HTML
    # -----------------------------------------------------

    share_html = f"""
    <style>

        .sov-section {{
            width: 100%;
            box-sizing: border-box;
            background: transparent;
            padding: 75px 7% 80px 7%;
        }}

        .sov-container {{
            width: min(980px, 100%);
            margin: 0 auto;
        }}

        .sov-kicker {{
            color: #C5253D;
            font-size: 12px;
            font-weight: 800;
            letter-spacing: 2.2px;
            text-transform: uppercase;
            margin-bottom: 9px;
        }}

        .sov-title {{
            color: #081A28;
            font-size: clamp(31px, 4vw, 48px);
            line-height: 1.05;
            font-weight: 850;
            margin: 0 0 42px 0;
        }}

        .sov-team-row {{
            display: grid;
            grid-template-columns: 180px 1fr 180px;
            align-items: center;
            gap: 22px;
        }}

        .sov-team {{
            display: flex;
            align-items: center;
            gap: 14px;
        }}

        .sov-team-right {{
            justify-content: flex-end;
        }}

        .sov-flag {{
            width: 68px;
            height: 45px;
            object-fit: cover;
            border-radius: 6px;
            border: 1px solid rgba(8, 26, 40, 0.18);
            box-shadow: 0 4px 12px rgba(8, 26, 40, 0.13);
            flex-shrink: 0;
        }}

        .sov-team-name {{
            color: #081A28;
            font-size: 17px;
            font-weight: 800;
        }}

        .sov-bar {{
            display: flex;
            width: 100%;
            height: 48px;
            overflow: hidden;
            border-radius: 8px;
            box-shadow: 0 5px 16px rgba(8, 26, 40, 0.14);
        }}

        .sov-segment {{
            min-width: 0;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #FFFFFF;
            font-size: 17px;
            font-weight: 850;
            transition: width 0.35s ease;
        }}

        .sov-segment-1 {{
            background: {color1};
            width: {percentage1}%;
        }}

        .sov-segment-2 {{
            background: {color2};
            width: {percentage2}%;
            border-left: 2px solid rgba(255, 255, 255, 0.85);
        }}

        .sov-mentions-row {{
            display: grid;
            grid-template-columns: 180px 1fr 180px;
            gap: 22px;
            margin-top: 10px;
        }}

        .sov-mentions {{
            color: #5A6872;
            font-size: 12px;
            font-weight: 650;
        }}

        .sov-mentions-right {{
            text-align: right;
        }}

        .sov-interpretation {{
            max-width: 720px;
            margin: 35px auto 0 auto;
            color: #344653;
            font-family: Georgia, "Times New Roman", serif;
            font-size: 17px;
            line-height: 1.65;
            text-align: center;
        }}

        .sov-unavailable {{
            padding: 24px 28px;
            color: #566570;
            background: rgba(8, 26, 40, 0.05);
            border-left: 4px solid #D5B536;
            font-size: 16px;
            line-height: 1.55;
        }}

        @media (max-width: 760px) {{

            .sov-section {{
                padding: 55px 6%;
            }}

            .sov-team-row {{
                grid-template-columns: 1fr 1fr;
                gap: 18px;
            }}

            .sov-team-right {{
                justify-content: flex-end;
            }}

            .sov-bar {{
                grid-column: 1 / -1;
                grid-row: 2;
            }}

            .sov-mentions-row {{
                display: none;
            }}

            .sov-team-name {{
                font-size: 15px;
            }}

            .sov-flag {{
                width: 58px;
                height: 38px;
            }}

        }}

    </style>

    <section class="sov-section">

        <div class="sov-container">

            <div class="sov-kicker">
                LA CONVERSACIÓN
            </div>

            <h2 class="sov-title">
                Quién dominó la conversación
            </h2>

            <div class="sov-team-row">

                <div class="sov-team">
                    {flag1}

                    <span class="sov-team-name">
                        {team1_es}
                    </span>
                </div>

                <div class="sov-bar">

                    <div class="sov-segment sov-segment-1">
                        {percentage1:.1f}%
                    </div>

                    <div class="sov-segment sov-segment-2">
                        {percentage2:.1f}%
                    </div>

                </div>

                <div class="sov-team sov-team-right">

                    <span class="sov-team-name">
                        {team2_es}
                    </span>

                    {flag2}

                </div>

            </div>

            <div class="sov-mentions-row">

                <div class="sov-mentions">
                    {mentions1:,} menciones
                </div>

                <div></div>

                <div class="sov-mentions sov-mentions-right">
                    {mentions2:,} menciones
                </div>

            </div>

            <div class="sov-interpretation">
                {interpretation}
            </div>

        </div>

    </section>
    """

    st.html(
        dedent(
            share_html
        ).strip()
    )

Appending to match_components.py


In [ ]:
%%writefile -a match_components.py

# =========================================================
# CLIMA EMOCIONAL
# =========================================================

EMOTION_ORDER_APP = [
    "Euforia",
    "Tensión",
    "Conflicto",
    "Tristeza"
]

EMOTION_COLORS_APP = {
    "Euforia": "#2ECC71",
    "Tensión": "#F1C40F",
    "Conflicto": "#E67E22",
    "Tristeza": "#3498DB"
}


def build_emotion_bar_html(
    summary
):
    """
    Construye una barra emocional apilada.
    """

    percentages = summary.get(
        "percentages",
        {}
    )

    segments = []

    for emotion in EMOTION_ORDER_APP:

        percentage = float(
            percentages.get(
                emotion,
                0
            )
        )

        if percentage <= 0:
            continue

        color = EMOTION_COLORS_APP[
            emotion
        ]

        # Los porcentajes muy pequeños no llevan texto
        # porque quedarían superpuestos.
        visible_label = (
            f"{percentage:.1f}%"
            if percentage >= 7
            else ""
        )

        text_color = (
            "#10212D"
            if emotion == "Tensión"
            else "#FFFFFF"
        )

        segments.append(
            f"""
            <div
                class="emotion-segment"
                style="
                    width: {percentage}%;
                    background: {color};
                    color: {text_color};
                "
                title="{emotion}: {percentage:.1f}%"
            >
                {visible_label}
            </div>
            """
        )

    return "".join(
        segments
    )


def render_emotional_climate(
    match_data,
    banderas_img
):
    """
    Muestra el clima emocional general y la comparación
    entre las conversaciones de ambos equipos.
    """

    emotional_data = match_data.get(
        "emotional_climate",
        {}
    )

    quality = match_data.get(
        "data_quality",
        {}
    )

    # -----------------------------------------------------
    # SECCIÓN SIN INFORMACIÓN EMOCIONAL
    # -----------------------------------------------------

    if (
    not quality.get(
        "show_social_analysis",
        False
    )
    or not emotional_data.get(
        "available",
        False
    )
):

      render_unavailable_section(
        kicker="LAS EMOCIONES",
        title="Clima emocional",
        message=(
            "No hay suficientes tuits para reconstruir "
            "el clima emocional de este partido."
        ),
        theme="warm"
    )

      return

    # -----------------------------------------------------
    # DATOS
    # -----------------------------------------------------

    overall = emotional_data.get(
        "overall",
        {}
    )

    team1_data = emotional_data.get(
        "team1",
        {}
    )

    team2_data = emotional_data.get(
        "team2",
        {}
    )

    team1 = team1_data.get(
        "name",
        ""
    )

    team2 = team2_data.get(
        "name",
        ""
    )

    team1_es = get_team_name_es(
        team1
    )

    team2_es = get_team_name_es(
        team2
    )

    team1_summary = team1_data.get(
        "summary",
        {}
    )

    team2_summary = team2_data.get(
        "summary",
        {}
    )

    flag1 = obtener_bandera_html(
        team1_es,
        banderas_img,
        css_class="emotion-flag"
    )

    flag2 = obtener_bandera_html(
        team2_es,
        banderas_img,
        css_class="emotion-flag"
    )

    overall_bar = build_emotion_bar_html(
        overall
    )

    team1_bar = build_emotion_bar_html(
        team1_summary
    )

    team2_bar = build_emotion_bar_html(
        team2_summary
    )

    interpretation = emotional_data.get(
        "interpretation",
        ""
    )

    classification_rate = float(
        overall.get(
            "classification_rate",
            0
        )
    )

    classified_tweets = int(
        overall.get(
            "classified_tweets",
            0
        )
    )

    # -----------------------------------------------------
    # LEYENDA
    # -----------------------------------------------------

    legend_items = []

    for emotion in EMOTION_ORDER_APP:

        legend_items.append(
            f"""
            <div class="emotion-legend-item">

                <span
                    class="emotion-legend-color"
                    style="
                        background:
                        {EMOTION_COLORS_APP[emotion]};
                    "
                ></span>

                <span>
                    {emotion}
                </span>

            </div>
            """
        )

    legend_html = "".join(
        legend_items
    )

    # -----------------------------------------------------
    # HTML
    # -----------------------------------------------------

    emotion_html = f"""
    <style>

        .emotion-section {{
            width: 100%;
            box-sizing: border-box;
            background: #F4F1EA;
            padding: 80px 7% 85px 7%;
        }}

        .emotion-container {{
            width: min(980px, 100%);
            margin: 0 auto;
        }}

        .emotion-kicker {{
            color: #C5253D;
            font-size: 12px;
            font-weight: 800;
            letter-spacing: 2.2px;
            text-transform: uppercase;
            margin-bottom: 9px;
        }}

        .emotion-title {{
            color: #081A28;
            font-size: clamp(31px, 4vw, 48px);
            line-height: 1.05;
            font-weight: 850;
            margin: 0;
        }}

        .emotion-legend {{
            display: flex;
            flex-wrap: wrap;
            gap: 14px 25px;
            margin: 30px 0 42px 0;
        }}

        .emotion-legend-item {{
            display: flex;
            align-items: center;
            gap: 8px;
            color: #344653;
            font-size: 14px;
            font-weight: 750;
        }}

        .emotion-legend-color {{
            width: 14px;
            height: 14px;
            border-radius: 3px;
            flex-shrink: 0;
        }}

        .emotion-chart {{
            display: flex;
            flex-direction: column;
            gap: 25px;
        }}

        .emotion-row {{
            display: grid;
            grid-template-columns: 155px 1fr;
            align-items: center;
            gap: 24px;
        }}

        .emotion-row-label {{
            display: flex;
            align-items: center;
            gap: 12px;
            color: #081A28;
            font-size: 16px;
            font-weight: 800;
        }}

        .emotion-overall-label {{
            padding-left: 5px;
        }}

        .emotion-flag {{
            width: 53px;
            height: 35px;
            object-fit: cover;
            border-radius: 5px;
            border: 1px solid rgba(8, 26, 40, 0.18);
            box-shadow: 0 3px 9px rgba(8, 26, 40, 0.12);
            flex-shrink: 0;
        }}

        .emotion-bar {{
            display: flex;
            width: 100%;
            height: 39px;
            overflow: hidden;
            border-radius: 6px;
            background: rgba(8, 26, 40, 0.08);
            box-shadow: 0 3px 10px rgba(8, 26, 40, 0.09);
        }}

        .emotion-segment {{
            display: flex;
            align-items: center;
            justify-content: center;
            min-width: 0;
            height: 100%;
            box-sizing: border-box;
            font-size: 12px;
            font-weight: 850;
            border-right: 1px solid rgba(255, 255, 255, 0.35);
        }}

        .emotion-segment:last-child {{
            border-right: none;
        }}

        .emotion-reading {{
            max-width: 760px;
            margin: 42px auto 0 auto;
            padding-top: 28px;
            border-top: 1px solid rgba(8, 26, 40, 0.17);
            text-align: center;
        }}

        .emotion-interpretation {{
            color: #243947;
            font-family: Georgia, "Times New Roman", serif;
            font-size: 18px;
            line-height: 1.65;
        }}

        .emotion-method {{
            margin-top: 10px;
            color: #6A777F;
            font-size: 12px;
            line-height: 1.45;
        }}

        .emotion-unavailable {{
            margin-top: 35px;
            padding: 24px 28px;
            color: #566570;
            background: rgba(8, 26, 40, 0.05);
            border-left: 4px solid #D5B536;
            font-size: 16px;
            line-height: 1.55;
        }}

        @media (max-width: 700px) {{

            .emotion-section {{
                padding: 58px 6%;
            }}

            .emotion-row {{
                grid-template-columns: 1fr;
                gap: 10px;
            }}

            .emotion-chart {{
                gap: 28px;
            }}

            .emotion-bar {{
                height: 35px;
            }}

            .emotion-legend {{
                margin-bottom: 34px;
            }}

        }}

    </style>

    <section class="emotion-section">

        <div class="emotion-container">

            <div class="emotion-kicker">
                LAS EMOCIONES
            </div>

            <h2 class="emotion-title">
                Clima emocional
            </h2>

            <div class="emotion-legend">
                {legend_html}
            </div>

            <div class="emotion-chart">

                <div class="emotion-row">

                    <div class="
                        emotion-row-label
                        emotion-overall-label
                    ">
                        Partido
                    </div>

                    <div class="emotion-bar">
                        {overall_bar}
                    </div>

                </div>

                <div class="emotion-row">

                    <div class="emotion-row-label">
                        {flag1}
                        <span>{team1_es}</span>
                    </div>

                    <div class="emotion-bar">
                        {team1_bar}
                    </div>

                </div>

                <div class="emotion-row">

                    <div class="emotion-row-label">
                        {flag2}
                        <span>{team2_es}</span>
                    </div>

                    <div class="emotion-bar">
                        {team2_bar}
                    </div>

                </div>

            </div>

            <div class="emotion-reading">

                <div class="emotion-interpretation">
                    {interpretation}
                </div>

                <div class="emotion-method">
                    {classified_tweets:,} tuits contenían
                    señales emocionales:
                    {classification_rate:.1f}% de la muestra analizada.
                </div>

            </div>

        </div>

    </section>
    """

    st.html(
        dedent(
            emotion_html
        ).strip()
    )

Appending to match_components.py


In [ ]:
%%writefile -a match_components.py

# =========================================================
# PROTAGONISTAS
# =========================================================

def render_protagonists(
    match_data,
    banderas_img
):
    """
    Muestra el ranking de los jugadores más mencionados.
    """

    protagonists = match_data.get(
        "protagonists",
        {}
    )

    quality = match_data.get(
        "data_quality",
        {}
    )

    ranking = protagonists.get(
        "ranking",
        []
    )

    # -----------------------------------------------------
    # EQUIPOS Y COLORES
    # -----------------------------------------------------

    hero = match_data.get(
        "hero",
        {}
    )

    team1 = hero.get(
        "team1",
        {}
    ).get(
        "name",
        ""
    )

    team2 = hero.get(
        "team2",
        {}
    ).get(
        "name",
        ""
    )

    color1, color2 = get_match_colors(
        match_data.get(
            "match",
            ""
        ),
        team1,
        team2
    )

    # -----------------------------------------------------
    # SECCIÓN SIN INFORMACIÓN
    # -----------------------------------------------------

    if (
    not quality.get(
        "show_social_analysis",
        False
    )
    or not protagonists.get(
        "available",
        False
    )
    or not ranking
):

      render_unavailable_section(
        kicker="LOS NOMBRES DEL PARTIDO",
        title="Protagonistas",
        message=(
            "No hay suficientes menciones para construir "
            "un ranking de protagonistas."
        ),
        theme="dark"
    )

      return

    # -----------------------------------------------------
    # FILAS DEL RANKING
    # -----------------------------------------------------

    max_mentions = max(
        int(
            player.get(
                "mentions",
                0
            )
        )
        for player in ranking
    )

    rows_html = []

    for player_data in ranking:

        rank = int(
            player_data.get(
                "rank",
                0
            )
        )

        player = player_data.get(
            "player",
            ""
        )

        team = player_data.get(
            "team",
            ""
        )

        mentions = int(
            player_data.get(
                "mentions",
                0
            )
        )

        team_es = get_team_name_es(
            team
        )

        flag = obtener_bandera_html(
            team_es,
            banderas_img,
            css_class="players-flag"
        )

        player_color = (
            color1
            if team == team1
            else color2
        )

        if max_mentions > 0:

            relative_width = round(
                mentions
                / max_mentions
                * 100,
                1
            )

        else:
            relative_width = 0

        rows_html.append(
            f"""
            <div class="players-row">

                <div class="players-rank">
                    {rank:02d}
                </div>

                <div class="players-identity">

                    {flag}

                    <div class="players-name-block">

                        <div class="players-name">
                            {player}
                        </div>

                        <div class="players-team">
                            {team_es}
                        </div>

                    </div>

                </div>

                <div class="players-bar-track">

                    <div
                        class="players-bar-fill"
                        style="
                            width: {relative_width}%;
                            background: {player_color};
                        "
                    ></div>

                </div>

                <div class="players-count">

                    <strong>
                        {mentions:,}
                    </strong>

                    <span>
                        menciones
                    </span>

                </div>

            </div>
            """
        )

    ranking_html = "".join(
        rows_html
    )

    interpretation = protagonists.get(
        "interpretation",
        ""
    )

    # -----------------------------------------------------
    # HTML
    # -----------------------------------------------------

    players_html = f"""
    <style>

        .players-section {{
            width: 100%;
            box-sizing: border-box;
            background:
                radial-gradient(
                    circle at top left,
                    #194564 0%,
                    #0B2232 42%,
                    #06131D 100%
                );
            padding: 82px 7% 88px 7%;
        }}

        .players-container {{
            width: min(980px, 100%);
            margin: 0 auto;
        }}

        .players-kicker {{
            color: #E1C33A;
            font-size: 12px;
            font-weight: 800;
            letter-spacing: 2.2px;
            text-transform: uppercase;
            margin-bottom: 9px;
        }}

        .players-title {{
            color: #FFFFFF;
            font-size: clamp(31px, 4vw, 48px);
            line-height: 1.05;
            font-weight: 850;
            margin: 0 0 45px 0;
        }}

        .players-ranking {{
            display: flex;
            flex-direction: column;
        }}

        .players-row {{
            display: grid;
            grid-template-columns:
                42px
                235px
                minmax(190px, 1fr)
                105px;
            align-items: center;
            gap: 20px;
            min-height: 79px;
            border-top:
                1px solid rgba(255, 255, 255, 0.13);
        }}

        .players-row:last-child {{
            border-bottom:
                1px solid rgba(255, 255, 255, 0.13);
        }}

        .players-rank {{
            color: #E1C33A;
            font-size: 14px;
            font-weight: 850;
            letter-spacing: 1px;
        }}

        .players-identity {{
            display: flex;
            align-items: center;
            gap: 14px;
            min-width: 0;
        }}

        .players-flag {{
            width: 48px;
            height: 32px;
            object-fit: cover;
            border-radius: 4px;
            border:
                1px solid rgba(255, 255, 255, 0.35);
            flex-shrink: 0;
        }}

        .players-name-block {{
            min-width: 0;
        }}

        .players-name {{
            color: #FFFFFF;
            font-size: 18px;
            font-weight: 820;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
        }}

        .players-team {{
            margin-top: 3px;
            color: rgba(255, 255, 255, 0.58);
            font-size: 11px;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.8px;
        }}

        .players-bar-track {{
            width: 100%;
            height: 13px;
            overflow: hidden;
            background: rgba(255, 255, 255, 0.12);
            border-radius: 20px;
        }}

        .players-bar-fill {{
            height: 100%;
            min-width: 3px;
            border-radius: 20px;
        }}

        .players-count {{
            color: #FFFFFF;
            text-align: right;
        }}

        .players-count strong {{
            display: block;
            font-size: 17px;
            font-weight: 850;
        }}

        .players-count span {{
            display: block;
            margin-top: 2px;
            color: rgba(255, 255, 255, 0.57);
            font-size: 10px;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}

        .players-interpretation {{
            max-width: 730px;
            margin: 38px auto 0 auto;
            color: rgba(255, 255, 255, 0.85);
            font-family: Georgia, "Times New Roman", serif;
            font-size: 18px;
            line-height: 1.65;
            text-align: center;
        }}

        .players-unavailable {{
            padding: 24px 28px;
            color: rgba(255, 255, 255, 0.74);
            background: rgba(255, 255, 255, 0.07);
            border-left: 4px solid #E1C33A;
            font-size: 16px;
            line-height: 1.55;
        }}

        @media (max-width: 760px) {{

            .players-section {{
                padding: 58px 6%;
            }}

            .players-row {{
                grid-template-columns:
                    30px
                    minmax(145px, 1fr)
                    75px;
                gap: 13px;
            }}

            .players-bar-track {{
                display: none;
            }}

            .players-flag {{
                width: 41px;
                height: 28px;
            }}

            .players-name {{
                font-size: 15px;
            }}

            .players-count strong {{
                font-size: 15px;
            }}

        }}

    </style>

    <section class="players-section">

        <div class="players-container">

            <div class="players-kicker">
                LOS NOMBRES DEL PARTIDO
            </div>

            <h2 class="players-title">
                Protagonistas
            </h2>

            <div class="players-ranking">
                {ranking_html}
            </div>

            <div class="players-interpretation">
                {interpretation}
            </div>

        </div>

    </section>
    """

    st.html(
        dedent(
            players_html
        ).strip()
    )

Appending to match_components.py


In [ ]:
%%writefile -a match_components.py

# =========================================================
# TWEETS DESTACADOS
# =========================================================

def format_social_number(
    value
):
    """
    Formatea números con separador de miles en español.
    """

    return (
        f"{int(value):,}"
        .replace(
            ",",
            "."
        )
    )


def render_featured_tweets(
    match_data
):
    """
    Muestra las tres piezas editoriales seleccionadas
    para representar la conversación del partido.
    """

    featured = match_data.get(
        "featured_tweets",
        {}
    )

    quality = match_data.get(
        "data_quality",
        {}
    )

    tweets = featured.get(
        "tweets",
        []
    )

    # Si la muestra no permite una selección editorial
    # sólida, la sección no se muestra.
    if (
        not quality.get(
            "show_featured_tweets",
            False
        )
        or not featured.get(
            "available",
            False
        )
        or not tweets
    ):
        return

    cards_html = []

    card_accents = [
        "#C5253D",
        "#D5B536",
        "#2B67D9"
    ]

    for index, tweet in enumerate(
        tweets[:3]
    ):

        label = tweet.get(
            "label",
            "Tweet destacado"
        )

        # En la aplicación se prioriza la traducción
        # editorial al español.
        text = (
            tweet.get(
                "translation_es"
            )
            or tweet.get(
                "text",
                ""
            )
        )

        likes = int(
            tweet.get(
                "likes",
                0
            )
        )

        retweets = int(
            tweet.get(
                "retweets",
                0
            )
        )

        accent = card_accents[
            index % len(
                card_accents
            )
        ]

        likes_html = ""

        if likes > 0:

            likes_html = f"""
            <span class="tweet-stat">
                ♥ {format_social_number(likes)}
            </span>
            """

        retweets_html = ""

        if retweets > 0:

            retweets_html = f"""
            <span class="tweet-stat">
                ↻ {format_social_number(retweets)}
            </span>
            """

        cards_html.append(
            f"""
            <article
                class="tweet-card"
                style="
                    border-top-color: {accent};
                "
            >

                <div class="tweet-card-label">
                    {label}
                </div>

                <blockquote class="tweet-card-text">
                    “{text}”
                </blockquote>

                <div class="tweet-card-footer">
                    {likes_html}
                    {retweets_html}
                </div>

            </article>
            """
        )

    cards_content = "".join(
        cards_html
    )

    tweets_html = f"""
    <style>

        .featured-section {{
            width: 100%;
            box-sizing: border-box;
            background: #FFFFFF;
            padding: 82px 7% 90px 7%;
        }}

        .featured-container {{
            width: min(1080px, 100%);
            margin: 0 auto;
        }}

        .featured-kicker {{
            color: #C5253D;
            font-size: 12px;
            font-weight: 800;
            letter-spacing: 2.2px;
            text-transform: uppercase;
            margin-bottom: 9px;
        }}

        .featured-title {{
            color: #081A28;
            font-size: clamp(31px, 4vw, 48px);
            line-height: 1.05;
            font-weight: 850;
            margin: 0 0 46px 0;
        }}

        .featured-grid {{
            display: grid;
            grid-template-columns:
                repeat(3, minmax(0, 1fr));
            gap: 22px;
            align-items: stretch;
        }}

        .tweet-card {{
            min-height: 285px;
            box-sizing: border-box;
            display: flex;
            flex-direction: column;
            padding: 28px 27px 23px 27px;
            background: #F4F1EA;
            border-top: 6px solid;
            border-radius: 4px 4px 12px 12px;
            box-shadow:
                0 8px 24px rgba(8, 26, 40, 0.10);
        }}

        .tweet-card-label {{
            color: #52616B;
            font-size: 11px;
            font-weight: 850;
            letter-spacing: 0.9px;
            text-transform: uppercase;
        }}

        .tweet-card-text {{
            flex: 1;
            margin: 24px 0;
            padding: 0;
            border: none;
            color: #102938;
            font-family:
                Georgia,
                "Times New Roman",
                serif;
            font-size: 18px;
            line-height: 1.58;
        }}

        .tweet-card-footer {{
            min-height: 20px;
            display: flex;
            align-items: center;
            gap: 18px;
            padding-top: 17px;
            border-top:
                1px solid rgba(8, 26, 40, 0.13);
        }}

        .tweet-stat {{
            color: #60717B;
            font-size: 12px;
            font-weight: 750;
        }}

        @media (max-width: 850px) {{

            .featured-grid {{
                grid-template-columns: 1fr;
            }}

            .tweet-card {{
                min-height: 220px;
            }}

        }}

        @media (max-width: 600px) {{

            .featured-section {{
                padding: 58px 6% 65px 6%;
            }}

            .tweet-card {{
                min-height: 0;
            }}

            .tweet-card-text {{
                font-size: 17px;
            }}

        }}

    </style>

    <section class="featured-section">

        <div class="featured-container">

            <div class="featured-kicker">
                LAS VOCES DEL PARTIDO
            </div>

            <h2 class="featured-title">
                Tweets destacados
            </h2>

            <div class="featured-grid">
                {cards_content}
            </div>

        </div>

    </section>
    """

    st.html(
        dedent(
            tweets_html
        ).strip()
    )

Appending to match_components.py


In [ ]:
%%writefile -a match_components.py

# =========================================================
# RADIOGRAFÍA FINAL
# =========================================================

def render_final_radiography(
    match_data
):
    """
    Muestra el resumen ejecutivo final del partido.
    """

    radiography = match_data.get(
        "radiography",
        {}
    )

    cards = radiography.get(
        "cards",
        []
    )

    final_sentence = radiography.get(
        "final_sentence",
        ""
    )

    if not cards:
        return

    card_accents = {
        "result": "#C5253D",
        "tweets": "#D5B536",
        "conversation": "#2B67D9",
        "protagonist": "#74A9D3",
        "emotion": "#2ECC71",
        "peak": "#E67E22"
    }

    cards_html = []

    for card in cards:

        key = card.get(
            "key",
            ""
        )

        label = card.get(
            "label",
            ""
        )

        value = card.get(
            "value",
            ""
        )

        detail = card.get(
            "detail",
            ""
        )

        accent = card_accents.get(
            key,
            "#D5B536"
        )

        unavailable = (
            "sin datos" in str(value).lower()
            or "sin cobertura" in str(value).lower()
        )

        unavailable_class = (
            " radiography-card-unavailable"
            if unavailable
            else ""
        )

        detail_html = ""

        if detail:

            detail_html = f"""
            <div class="radiography-card-detail">
                {detail}
            </div>
            """

        cards_html.append(
            f"""
            <article
                class="
                    radiography-card
                    {unavailable_class}
                "
                style="
                    --card-accent: {accent};
                "
            >

                <div class="radiography-card-line"></div>

                <div class="radiography-card-label">
                    {label}
                </div>

                <div class="radiography-card-value">
                    {value}
                </div>

                {detail_html}

            </article>
            """
        )

    cards_content = "".join(
        cards_html
    )

    radiography_html = f"""
    <style>

        .radiography-section {{
            position: relative;
            width: 100%;
            box-sizing: border-box;
            overflow: hidden;
            background:
                radial-gradient(
                    circle at top,
                    #244B64 0%,
                    #0B2232 43%,
                    #040B10 100%
                );
            padding: 88px 7% 105px 7%;
        }}

        .radiography-section::after {{
            content: "";
            position: absolute;
            left: 0;
            right: 0;
            bottom: 0;
            height: 7px;
            background:
                linear-gradient(
                    90deg,
                    #C5253D 0%,
                    #C5253D 33.33%,
                    #2B67D9 33.33%,
                    #2B67D9 66.66%,
                    #D5B536 66.66%,
                    #D5B536 100%
                );
        }}

        .radiography-container {{
            width: min(1080px, 100%);
            margin: 0 auto;
        }}

        .radiography-kicker {{
            color: #D5B536;
            font-size: 12px;
            font-weight: 800;
            letter-spacing: 2.2px;
            text-transform: uppercase;
            margin-bottom: 9px;
        }}

        .radiography-title {{
            color: #FFFFFF;
            font-size: clamp(32px, 4vw, 50px);
            line-height: 1.05;
            font-weight: 850;
            margin: 0 0 48px 0;
        }}

        .radiography-grid {{
            display: grid;
            grid-template-columns:
                repeat(3, minmax(0, 1fr));
            gap: 18px;
        }}

        .radiography-card {{
            position: relative;
            min-height: 165px;
            box-sizing: border-box;
            overflow: hidden;
            padding: 27px 25px 24px 25px;
            background: rgba(255, 255, 255, 0.075);
            border:
                1px solid rgba(255, 255, 255, 0.13);
            border-radius: 10px;
        }}

        .radiography-card-line {{
            position: absolute;
            top: 0;
            left: 0;
            width: 100%;
            height: 5px;
            background: var(--card-accent);
        }}

        .radiography-card-label {{
            color: rgba(255, 255, 255, 0.57);
            font-size: 10px;
            font-weight: 850;
            letter-spacing: 1px;
            text-transform: uppercase;
        }}

        .radiography-card-value {{
            margin-top: 18px;
            color: #FFFFFF;
            font-size: clamp(21px, 2.3vw, 30px);
            line-height: 1.13;
            font-weight: 850;
        }}

        .radiography-card-detail {{
            margin-top: 12px;
            color: rgba(255, 255, 255, 0.67);
            font-size: 12px;
            font-weight: 650;
            line-height: 1.4;
        }}

        .radiography-card-unavailable {{
            opacity: 0.52;
        }}

        .radiography-ending {{
            max-width: 850px;
            margin: 58px auto 0 auto;
            padding-top: 38px;
            border-top:
                1px solid rgba(255, 255, 255, 0.18);
            color: #FFFFFF;
            font-family:
                Georgia,
                "Times New Roman",
                serif;
            font-size: clamp(22px, 2.7vw, 32px);
            line-height: 1.5;
            text-align: center;
        }}

        @media (max-width: 850px) {{

            .radiography-grid {{
                grid-template-columns:
                    repeat(2, minmax(0, 1fr));
            }}

        }}

        @media (max-width: 580px) {{

            .radiography-section {{
                padding: 60px 6% 78px 6%;
            }}

            .radiography-grid {{
                grid-template-columns: 1fr;
            }}

            .radiography-card {{
                min-height: 145px;
            }}

        }}

    </style>

    <section class="radiography-section">

        <div class="radiography-container">

            <div class="radiography-kicker">
                EL PARTIDO EN SEIS CLAVES
            </div>

            <h2 class="radiography-title">
                Radiografía final
            </h2>

            <div class="radiography-grid">
                {cards_content}
            </div>

            <div class="radiography-ending">
                {final_sentence}
            </div>

        </div>

    </section>
    """

    st.html(
        dedent(
            radiography_html
        ).strip()
    )

Appending to match_components.py


In [ ]:
%%writefile -a match_components.py

# =========================================================
# AVISO EDITORIAL PARA SECCIONES SIN DATOS
# =========================================================

def render_unavailable_section(
    kicker,
    title,
    message,
    theme="light"
):
    """
    Muestra una sección editorial cuando un análisis
    específico no se encuentra disponible.
    """

    if theme == "dark":

        background = """
            radial-gradient(
                circle at top left,
                #194564 0%,
                #0B2232 42%,
                #06131D 100%
            )
        """

        title_color = "#FFFFFF"
        text_color = "rgba(255,255,255,0.78)"
        kicker_color = "#E1C33A"
        notice_background = "rgba(255,255,255,0.07)"
        notice_border = "rgba(255,255,255,0.14)"

    elif theme == "warm":

        background = "#F4F1EA"
        title_color = "#081A28"
        text_color = "#52616B"
        kicker_color = "#C5253D"
        notice_background = "rgba(8,26,40,0.05)"
        notice_border = "rgba(8,26,40,0.13)"

    else:

        background = "#FFFFFF"
        title_color = "#081A28"
        text_color = "#52616B"
        kicker_color = "#C5253D"
        notice_background = "#F4F1EA"
        notice_border = "rgba(8,26,40,0.13)"

    unavailable_html = f"""
    <style>

        .unavailable-section {{
            width: 100%;
            box-sizing: border-box;
            background: {background};
            padding: 72px 7% 76px 7%;
        }}

        .unavailable-container {{
            width: min(980px, 100%);
            margin: 0 auto;
        }}

        .unavailable-kicker {{
            color: {kicker_color};
            font-size: 12px;
            font-weight: 800;
            letter-spacing: 2.2px;
            text-transform: uppercase;
            margin-bottom: 9px;
        }}

        .unavailable-title {{
            color: {title_color};
            font-size: clamp(30px, 4vw, 46px);
            line-height: 1.05;
            font-weight: 850;
            margin: 0 0 34px 0;
        }}

        .unavailable-notice {{
            padding: 23px 27px;
            color: {text_color};
            background: {notice_background};
            border:
                1px solid {notice_border};
            border-left:
                5px solid {kicker_color};
            border-radius: 7px;
            font-size: 16px;
            font-weight: 600;
            line-height: 1.55;
        }}

        @media (max-width: 600px) {{

            .unavailable-section {{
                padding: 53px 6% 57px 6%;
            }}

        }}

    </style>

    <section class="unavailable-section">

        <div class="unavailable-container">

            <div class="unavailable-kicker">
                {kicker}
            </div>

            <h2 class="unavailable-title">
                {title}
            </h2>

            <div class="unavailable-notice">
                {message}
            </div>

        </div>

    </section>
    """

    st.html(
        dedent(
            unavailable_html
        ).strip()
    )

Appending to match_components.py


In [ ]:
!python -m py_compile match_components.py

In [ ]:
%%writefile app.py

import streamlit as st
import base64
import glob
from textwrap import dedent
from banderas import cargar_banderas, obtener_bandera_html
from data_loader import (
    load_match_index,
    load_match_package,
    get_matches_by_stage,
    MatchDataError
)

from app_config import (
    STAGE_CONFIG,
    get_team_name_es
)
from match_components import (
    render_match_hero,
    render_match_story,
    render_share_of_voice,
    render_emotional_climate,
    render_protagonists,
    render_featured_tweets,
    render_final_radiography
)
from match_charts import render_twitter_momentum
# =========================================================
# CONFIGURACIÓN
# =========================================================

st.set_page_config(
    page_title="Mundial Rusia 2018 desde Twitter",
    page_icon="⚽",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# =========================================================
# ESTILOS GENERALES
# =========================================================

st.markdown(
    dedent("""
    <style>
    #MainMenu { visibility: hidden; }
    header { visibility: hidden; }
    footer { visibility: hidden; }
    [data-testid="stSidebar"] { display: none; }
    .block-container { padding: 0rem !important; max-width: 100% !important; }
    div[data-testid="stVerticalBlock"] { gap: 0rem; }
    body { background-color: #06131d; }

    .btn-custom {
        display: inline-block;
        background-color: #87CEEB;
        color: #003366 !important;
        padding: 15px 45px;
        border-radius: 12px;
        text-decoration: none !important;
        font-weight: bold;
        font-size: 19px;
        margin-top: 30px;
        transition: 0.25s;
        box-shadow: 0 5px 18px rgba(0,0,0,0.40);
    }
    .btn-custom:hover { background-color: #B0E2FF; transform: translateY(-3px); }

    .btn-volver {
        display: inline-block;
        color: #87CEEB !important;
        text-decoration: none !important;
        font-size: 16px;
        font-weight: bold;
        margin-top: 35px;
    }

    .pagina {
        min-height: 100vh;
        box-sizing: border-box;
        background: radial-gradient(circle at top, #194564 0%, #081a28 48%, #020609 100%);
        color: white;
        padding: 65px 7%;
    }

    .grilla-fases { display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 20px; }
    .tarjeta-fase {
        display: block; padding: 32px 20px; border-radius: 18px; text-align: center;
        text-decoration: none !important; background: rgba(255,255,255,0.08);
        border: 1px solid rgba(255,255,255,0.14); transition: 0.25s; color: white !important;
    }
    .tarjeta-fase:hover { background: rgba(135,206,235,0.18); border-color: #87CEEB; transform: translateY(-5px); }
    .tarjeta-fase.activa { background: rgba(135,206,235,0.22); border-color: #87CEEB; }

    .seccion-partidos { margin-top: 55px; }
    .titulo-partidos { color: white; text-align: center; font-size: 30px; font-weight: 800; margin-bottom: 28px; }
    .grilla-partidos { display: grid; grid-template-columns: repeat(2, minmax(280px, 1fr)); gap: 18px; max-width: 1000px; margin: 0 auto; }
    .tarjeta-partido {
        min-height: 90px; display: flex; justify-content: center; align-items: center; box-sizing: border-box; padding: 22px;
        color: white !important; text-decoration: none !important; text-align: center; font-size: 21px; font-weight: 750;
        background: rgba(255,255,255,0.075); border: 1px solid rgba(255,255,255,0.14); border-radius: 16px; transition: 0.25s;
    }
    .tarjeta-partido:hover { color: #87CEEB !important; background: rgba(135,206,235,0.15); border-color: #87CEEB; transform: translateY(-3px); }

    .pantalla-partido { min-height: 100vh; box-sizing: border-box; padding: 65px 7%; color: white; background: radial-gradient(circle at top, #194564 0%, #081a28 48%, #020609 100%); }
    .nombre-partido-seleccionado { margin-top: 120px; color: white; text-align: center; font-size: 52px; font-weight: 850; }

    @media (max-width: 700px) {
        .grilla-partidos { grid-template-columns: 1fr; }
        .nombre-partido-seleccionado { font-size: 38px; }
    }
    </style>
    """),
    unsafe_allow_html=True
)

# =========================================================
# CARGAR IMÁGENES
# =========================================================

def cargar_imagen_base64(patrones):

    for patron in patrones:

        archivos = glob.glob(
            patron,
            recursive=True
        )

        if archivos:

            ruta = archivos[0]
            extension = ruta.lower().split(".")[-1]

            if extension == "jpg":
                extension = "jpeg"

            if extension in ["webp", "png", "jpeg"]:

                with open(ruta, "rb") as archivo:

                    imagen_base64 = base64.b64encode(
                        archivo.read()
                    ).decode()

                return imagen_base64, extension

    return "", "jpeg"


# Imagen de la portada: pelota y copa

img_b64, ext_img = cargar_imagen_base64([
    "/content/*pelota*.*",
    "/content/drive/MyDrive/**/*pelota*.*"
])


# Imagen de la pantalla de fases: estadio

estadio_b64, ext_estadio = cargar_imagen_base64([
    "/content/drive/MyDrive/**/estadio-rusia-reuters.*"
])
banderas_img = cargar_banderas()
# =========================================================
# CARGAR ÍNDICE DE LOS 16 PARTIDOS
# =========================================================

try:

    match_index = load_match_index()

except MatchDataError as error:

    st.error(
        "No fue posible cargar los datos "
        "de los partidos."
    )

    st.code(
        str(error)
    )

    st.stop()
# =========================================================
# NAVEGACIÓN Y CONTENIDO
# =========================================================

pantalla = st.query_params.get("p", "inicio")
fase_activa = st.query_params.get("fase", "")
partido_activo = st.query_params.get("partido", "")

if pantalla == "inicio":
    fondo_css = f"background-image: linear-gradient(rgba(0,0,0,0.3), rgba(0,0,0,0.7)), url('data:image/{ext_img};base64,{img_b64}');" if img_b64 else "background: #06131d;"
    st.markdown(f"""
        <div style="{fondo_css} height: 100vh; background-size: cover; background-position: center; display: flex; flex-direction: column; justify-content: center; align-items: center; text-align: center; color: white; width: 100%;">
            <div style="max-width: 900px; padding: 20px;">
                <div style="letter-spacing: 4px; font-size: 14px; opacity: 0.8; margin-bottom: 20px;">SEMINARIO FIUBA 9216 • TWITTER MINING</div>
                <h1 style="font-size: clamp(40px, 8vw, 75px); font-weight: 850; margin: 0; line-height: 1.1;">MUNDIAL RUSIA 2018</h1>
                <h2 style="font-size: clamp(20px, 4vw, 40px); font-weight: 300; margin-bottom: 30px;">DESDE TWITTER</h2>
                <p style="font-size: 19px; opacity: 0.9;">La historia de las fases finales reconstruida a partir de más de 500.000 tweets.</p>
                <a href="?p=fases" target="_self" class="btn-custom">Explorar partidos</a>
            </div>
        </div>
    """, unsafe_allow_html=True)

elif pantalla == "fases":
      # =====================================================
    # ESTÉTICA DE LA PANTALLA DE FASES
    # =====================================================

      if estadio_b64:

        fondo_fases = (
            "linear-gradient("
            "rgba(0, 0, 0, 0.28), "
            "rgba(0, 0, 0, 0.65)"
            "), "
            f"url('data:image/{ext_estadio};base64,{estadio_b64}')"
        )

      else:

        fondo_fases = (
            "linear-gradient("
            "#2d2b2a, "
            "#121212"
            ")"
        )

      st.markdown(
        dedent(f"""
        <style>

        /* FONDO CON LA FOTO DEL ESTADIO */

        .pagina {{
            min-height: 100vh;
            box-sizing: border-box;

            background-image: {fondo_fases};
            background-size: cover;
            background-position: center;
            background-repeat: no-repeat;
            background-attachment: fixed;

            color: #20272d;
            padding: 58px 6%;
        }}


        /* TÍTULO PRINCIPAL */

        .pagina > div:first-child h1 {{
            color: #ffffff !important;
            font-size: 48px !important;
            font-weight: 800 !important;
            letter-spacing: -1px;
            text-shadow: 0 3px 14px rgba(0, 0, 0, 0.55);
        }}

        .pagina > div:first-child p {{
            color: rgba(255, 255, 255, 0.88) !important;
            font-size: 18px !important;
            text-shadow: 0 2px 8px rgba(0, 0, 0, 0.50);
        }}


        /* TARJETAS DE LAS FASES */

        .grilla-fases {{
            display: grid;
            grid-template-columns:
                repeat(auto-fit, minmax(190px, 1fr));

            gap: 16px;
            max-width: 1250px;
            margin: 0 auto;
        }}

        .tarjeta-fase {{
            min-height: 125px;
            box-sizing: border-box;

            display: flex;
            flex-direction: column;
            justify-content: center;

            padding: 24px 18px;
            border-radius: 12px;

            text-align: center;
            text-decoration: none !important;

            background: rgba(255, 255, 255, 0.90);
            border: 1px solid rgba(95, 69, 62, 0.20);

            box-shadow:
                0 8px 22px rgba(47, 39, 32, 0.11);

            backdrop-filter: blur(5px);
            transition: 0.22s ease;
        }}

        .tarjeta-fase h3 {{
            color: #792d3b !important;
            font-size: 19px;
            font-weight: 800;
            text-transform: uppercase;
            line-height: 1.2;
            margin: 0 0 10px 0;
        }}

        .tarjeta-fase p {{
            color: #756a63 !important;
            font-size: 12px;
            font-weight: 700;
            letter-spacing: 1.2px;
            text-transform: uppercase;
            margin: 0;
        }}

        .tarjeta-fase:hover {{
            background: rgba(255, 255, 255, 0.98);
            border-color: #9e3545;

            box-shadow:
                0 12px 28px rgba(47, 39, 32, 0.16);

            transform: translateY(-3px);
        }}

        .tarjeta-fase.activa {{
            background: rgba(255, 255, 255, 0.98);
            border: 2px solid #9e3545;

            box-shadow:
                0 10px 25px rgba(85, 34, 43, 0.18);
        }}


        /* TÍTULO DE LA FASE SELECCIONADA */

        .titulo-partidos {{
            color: #792d3b !important;
            text-align: left;
            font-size: 24px;
            font-weight: 850;
            text-transform: uppercase;

            margin-bottom: 22px;
            padding-left: 15px;

            border-left: 4px solid #c0a052;
        }}


        /* TARJETAS DE LOS PARTIDOS */

        .grilla-partidos {{
            display: grid;
            grid-template-columns:
                repeat(2, minmax(280px, 1fr));

            gap: 14px;
            max-width: 1100px;
            margin: 0 auto;
        }}

                .tarjeta-partido {{
            min-height: 72px;
            box-sizing: border-box;

            display: grid;
            grid-template-columns: 1fr auto 1fr;
            align-items: center;
            gap: 20px;

            padding: 18px 22px;
            border-radius: 10px;

            color: #27323a !important;
            text-decoration: none !important;
            text-align: center;

            font-size: 18px;
            font-weight: 750;

            background: rgba(255, 255, 255, 0.92);
            border: 1px solid rgba(95, 69, 62, 0.18);

            box-shadow:
                0 6px 18px rgba(47, 39, 32, 0.09);

            backdrop-filter: blur(5px);
            transition: 0.20s ease;
        }}

        .lado-partido {{
            display: flex;
            align-items: center;
            gap: 10px;
            color: #27323a;
        }}

        .lado-izquierdo {{
            justify-content: flex-end;
            text-align: right;
        }}

        .lado-derecho {{
            justify-content: flex-start;
            text-align: left;
        }}

        .bandera-partido {{
            width: 42px;
            height: 28px;
            object-fit: cover;
            flex-shrink: 0;

            border-radius: 3px;
            border: 1px solid rgba(0, 0, 0, 0.12);

            box-shadow:
                0 2px 6px rgba(0, 0, 0, 0.22);
        }}

        .versus-partido {{
            color: #9e3545;
            font-size: 12px;
            font-weight: 900;
            letter-spacing: 1.5px;
        }}

        .tarjeta-partido:hover {{
            color: #8c2939 !important;
            background: #ffffff;
            border-color: #9e3545;

            box-shadow:
                0 9px 22px rgba(47, 39, 32, 0.14);

            transform: translateY(-2px);
        }}


        /* BOTÓN PARA VOLVER */

        .pagina .btn-volver {{
            color: #792d3b !important;
            margin-top: 38px;
        }}

        .pagina .btn-volver:hover {{
            color: #4f1822 !important;
        }}


        /* VERSIÓN PARA CELULAR */

        @media (max-width: 700px) {{

            .pagina {{
                padding: 42px 20px;
                background-attachment: scroll;
            }}

            .pagina > div:first-child h1 {{
                font-size: 36px !important;
            }}

            .grilla-partidos {{
                grid-template-columns: 1fr;
            }}

            .tarjeta-fase {{
                min-height: 105px;
            }}
            .tarjeta-partido {{
    gap: 10px;
    padding-left: 12px;
    padding-right: 12px;
    font-size: 15px;
}}

            .bandera-partido {{
                width: 34px;
                height: 23px;
            }}
        }}

        </style>
        """).strip(),
        unsafe_allow_html=True
    )
            # =====================================================
      # FASES Y PARTIDOS DESDE index.json
      # =====================================================

      fases = {}

      for fase_slug, fase_config in (
          STAGE_CONFIG.items()
      ):

          partidos_de_la_fase = (
              get_matches_by_stage(
                  fase_config[
                      "stage_key"
                  ]
              )
          )

          partidos_visibles = []

          for match_record in (
              partidos_de_la_fase
          ):

              partidos_visibles.append(
                  (
                      match_record["slug"],

                      get_team_name_es(
                          match_record["team1"]
                      ),

                      get_team_name_es(
                          match_record["team2"]
                      )
                  )
              )

          fases[fase_slug] = {
              "nombre":
                  fase_config["nombre"],

              "cantidad":
                  fase_config["cantidad"],

              "partidos":
                  partidos_visibles
          }
      tarjetas_fases = ""
      for slug, datos in fases.items():
        clase = "tarjeta-fase activa" if fase_activa == slug else "tarjeta-fase"
        tarjetas_fases += (
            f'<a '
            f'href="?p=fases&amp;fase={slug}" '
            f'target="_self" '
            f'class="{clase}">'
                f'<h3>{datos["nombre"]}</h3>'
                f'<p>{datos["cantidad"]}</p>'
            f'</a>'
        )

      seccion_partidos = ""

      if fase_activa in fases:

          partidos_html = ""

          for partido_id, equipo1, equipo2 in fases[fase_activa]["partidos"]:

            bandera1 = obtener_bandera_html(
                equipo1,
                banderas_img
            )

            bandera2 = obtener_bandera_html(
                equipo2,
                banderas_img
            )

            partidos_html += (
                f'<a '
                f'href="?p=partido&amp;fase={fase_activa}&amp;partido={partido_id}" '
                f'target="_self" '
                f'class="tarjeta-partido">'

                    f'<span class="lado-partido lado-izquierdo">'
                        f'{bandera1}'
                        f'<span>{equipo1}</span>'
                    f'</span>'

                    f'<span class="versus-partido">'
                        f'VS'
                    f'</span>'

                    f'<span class="lado-partido lado-derecho">'
                        f'<span>{equipo2}</span>'
                        f'{bandera2}'
                    f'</span>'

                f'</a>'
            )

            seccion_partidos = (
            '<div class="seccion-partidos">'
                f'<div class="titulo-partidos">'
                    f'{fases[fase_activa]["nombre"]}'
                '</div>'
                f'<div class="grilla-partidos">'
                    f'{partidos_html}'
                '</div>'
            '</div>'
        )

      html_fases = (
        '<div class="pagina">'
            '<div style="text-align:center; margin-bottom:50px;">'
                '<h1 style="color:white; font-size:50px; margin-bottom:12px;">'
                    'Elegí una instancia'
                '</h1>'
                '<p style="color:#aec3d0; font-size:18px;">'
                    'Explorá cómo se vivieron los partidos decisivos.'
                '</p>'
            '</div>'
            f'<div class="grilla-fases">{tarjetas_fases}</div>'
            f'{seccion_partidos}'
            '<a href="?p=inicio" target="_self" class="btn-volver">'
                '← Volver al inicio'
            '</a>'
        '</div>'
    )

      st.markdown(
        html_fases,
        unsafe_allow_html=True
    )

elif pantalla == "partido":
        # =====================================================
    # CARGAR EL PARTIDO SELECCIONADO
    # =====================================================

    try:

        match_data = load_match_package(
            partido_activo
        )

    except MatchDataError as error:

        st.error(
            "No fue posible cargar "
            "el partido seleccionado."
        )

        st.code(
            str(error)
        )

        st.markdown(
            (
                f'<a '
                f'href="?p=fases&amp;fase={fase_activa}" '
                f'target="_self" '
                f'class="btn-volver">'
                f'← Volver a los partidos'
                f'</a>'
            ),
            unsafe_allow_html=True
        )

        st.stop()

    render_match_hero(
        match_data=match_data,
        banderas_img=banderas_img,
        fase_slug=fase_activa
    )
    render_match_story(
        match_data=match_data
    )
    render_twitter_momentum(
    match_data,
    banderas_img
)
    render_share_of_voice(
    match_data,
    banderas_img
)
    render_emotional_climate(
    match_data,
    banderas_img
)
    render_protagonists(
    match_data,
    banderas_img
)
    render_featured_tweets(
    match_data
)
    render_final_radiography(
    match_data
)

Overwriting app.py


In [ ]:
import os

archivos_necesarios = [
    "banderas.py",
    "data_loader.py",
    "app_config.py",
    "chart_config.py",
    "match_components.py",
    "match_charts.py",
    "app.py"
]

for archivo in archivos_necesarios:
    print(
        "✅" if os.path.exists(archivo) else "❌",
        archivo
    )

✅ banderas.py
✅ data_loader.py
✅ app_config.py
✅ chart_config.py
✅ match_components.py
✅ match_charts.py
✅ app.py


In [ ]:
# Cerrar una ejecución anterior, si existe
!pkill -f "streamlit run app.py" || true

# Iniciar la versión actualizada
!streamlit run app.py &>/content/log.txt &

^C


In [ ]:
from google.colab import userdata
from pyngrok import ngrok


# Leer el token desde los Secretos de Colab
token_ngrok = userdata.get(
    "NGROK_TOKEN"
)

if not token_ngrok:
    raise ValueError(
        "No se encontró el secreto NGROK_TOKEN"
    )


# Configurar ngrok sin mostrar el token
ngrok.set_auth_token(
    token_ngrok
)

# Cerrar túneles anteriores
ngrok.kill()

# Crear el enlace de la aplicación
tunnel = ngrok.connect(8501)

APP_URL = tunnel.public_url

print("✅ Aplicación disponible en:")
print(APP_URL)

✅ Aplicación disponible en:
https://amperage-runner-thinning.ngrok-free.dev
